#DERS 19- ADABOOST 

Rastgele binlerce ki?iye karma??k bir soru sordu?unuzu varsayal?m ve cevaplar?n? toplayal?m. ?o?u zaman, bu toplu cevab?n bir uzman?n cevab?ndan daha iyi oldu?unu g?receksiniz.Benzer ?ekilde, bir grubun tahminlerini (s?n?flay?c?lar veya regres?rler gibi) toplarsan?z, genellikle en iyi bireyselden daha iyi tahminler al?rs?n?z.Bir grup ?eklinde olan tahmin algoritmalar?na ensemble denir; bu nedenle, bu tekni?e Ensemble Learning denir ve bir Ensemble Learning algoritmas?na Ensemble y?ntemi denir.

Adaboost
Yeni bir predictor(tahminci) bir ?nceki tahmini d?zeltmesinin bir yolu, bir ?nceki tahminin uygulad??? e?itim durumlar?na biraz daha dikkat etmektir. Bu, zor durumlara giderek daha fazla odaklanan yeni tahminlerle sonu?lan?r. AdaBoost taraf?ndan kullan?lan teknik budur. ?rne?in, bir AdaBoost s?n?fland?r?c?s? olu?turmak i?in, birinci temel s?n?fland?r?c? e?itilir ve e?itim seti hakk?nda tahminlerde bulunmak i?in kullan?l?r. Yanl?? s?n?fland?r?lm?? e?itim durumlar?n?n nispi a??rl??? art?r?l?r ve bizim i?in daha de?erli hale gelir. ?kinci bir s?n?fland?r?c?, g?ncellenmi? a??rl?klar kullan?larak e?itilir ve yine antrenman setinde tahminler yapar, a??rl?klar g?ncellenir bu ?zellik son a?amaya kadar tekrar edilir.

T?m modeller e?itildikten sonra, ensemble tahmin yapmay?, bagging veya pastinge benzer ?ekilde yapar, ancak modeller, a??rl?kl? e?itim setindeki genel do?ruluklar?na ba?l? olarak farkl? a??rl?klara sahiptir.

Not: Bu s?ral? ??renme tekni?inin ?nemli bir dezavantaj? vard?r: Paralelle?tirilemez , ??nk? her tahminci yaln?zca ?nceki modele g?re e?itilip de?erlendirildikten sonra e?itilebilir. Sonu? olarak, bagging veya pasting gibi ?l?eklenmez.(?eekiller) 

Detayl? Okuma: https://medium.com/@oguzkircicek/ensemble-learning-7ec8c9a7227f

Bu ?al??mada, meme kanseri veri setini inceleyecek ve ki?inin meme kanseri olup olmad???n? tahmin etmek i?in modeli e?itmeye ?al??aca??z. Weak learner kullanaca??z yani maksimum derinli?i =2 olan bir karar a?ac? diyebiliriz.

Daha sonra weak learner'lar? 3'er 3'er artt?rarak en iyi modeli bulmaya ?al??aca??z.

K?t?phaneleri y?kleyerek ba?layal?m.

In [ ]:
import numpy as np 
import pandas as pd 
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_digits
from sklearn import metrics
%matplotlib inline

import os
import warnings
warnings.filterwarnings('ignore')


Meme kanseri veri setini kullanarak ki?inin kanser olup olmad???n? bulmaya ?al??aca??z. Ba??ml? de?i?kenimiz target.
A?a??daki veri setindeki ba??ms?z de?i?kenlere bakt???m?zda anlayabilmek i?in bir doktor olmam?z gerekebilirdi.
Ancak makine ??renmesinde veriler ile i? yapt???m?zdan bu i?in uzman? olmam?za gerek yok. Gerekli ?r?nt?y? yakalay?p modeli
kurdu?umuzda ba?ar?l? sonu?lar elde edebiliriz.

In [ ]:
cancer = load_breast_cancer() # verinin y?klenmesi
digits = load_digits()

data = cancer

In [ ]:
df = pd.DataFrame(data= np.c_[data['data'], data['target']],
                     columns= list(data['feature_names']) + ['target'])
df['target'] = df['target'].astype('uint16') # ba??ml? de?i?kenin veri tipinin belirlenmesi

In [ ]:
df

In [ ]:
df.head() #verinin ilk 5 sat?r?

In [ ]:
# ba??ms?z ve ba??ml? de?i?kenlerin birbirinden ay?r?lmas?
X = df.drop('target', axis=1)
y = df[['target']]

# verinin train ve test set'lere b?l?nmesi.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=101)
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

In [ ]:
# train ve test set'lerdeki ba??ml? de?i?kenin benzer ?ekilde da??l?p da??lmad???n?n kontrol edilmesi.
print(y_train.mean())
print(y_test.mean())

In [ ]:
# Adaboost'ta derinli?i olmayan ufak karar a?a?lar? kullan?yorduk. Maksimum k?r?l?m? 2 olan karar a?ac? olu?tural?m.
shallow_tree = DecisionTreeClassifier(max_depth=2, random_state = 100)

In [ ]:
# modeli ??renelim.
shallow_tree.fit(X_train, y_train)

# test hatalar?n?n ekrana bast?r?lmas?.
y_pred = shallow_tree.predict(X_test)
score = metrics.accuracy_score(y_test, y_pred)
score

?imdi de adaboost'ta 1'den 50'ye kadar 3'er 3'er artacak ?ekilde farkl? say?larda weak learner ile s?n?flama yapal?m.

In [ ]:
# olas? a?a? say?lar?n? tutacak olan listenin olu?turulmas?.

estimators = list(range(1, 50, 3))

abc_scores = []
for n_est in estimators:
    ABC = AdaBoostClassifier(
    base_estimator=shallow_tree, 
    n_estimators = n_est)
    
    ABC.fit(X_train, y_train)
    y_pred = ABC.predict(X_test)
    score = metrics.accuracy_score(y_test, y_pred)
    abc_scores.append(score)
    

In [ ]:
abc_scores # Farkl? adaboost modellerinin tutarl?l?k sonu?lar?.

In [ ]:
# test accuracy'lerinin plot'una bakarsak weak learner say?s? artt?k?a modelin test verisindeki ba?ar?s?n?n artt???n? s?yleyebiliriz.

plt.plot(estimators, abc_scores)
plt.xlabel('n_estimators')
plt.ylabel('accuracy')
plt.ylim([0.85, 1])
plt.show()